# SHREC 2021 Demo — Two-Stage Particle Detection and Official F1 Evaluation

This notebook runs the released SHREC pipeline on the example tomogram and reports the official-style F1 score. It loads the paired detector/classifier checkpoints, generates proposals, refines centers with the 3D CNN, and evaluates against the class mask.

Run cells in order from top to bottom. The example data is the SHREC 2021 model_9 tomogram with known ground truth.

## 1) Configure Paths and Imports

Set the release model and example data paths, then import the project modules. The release bundle contains `small/` and `large/` paired detector + classifier checkpoints; the example data includes the tomogram, class mask, and ground-truth labels.

In [23]:
import os
import sys

import numpy as np
import pandas as pd
import torch

sys.path.append("../src")

import data
import modules
import postprocess
import utils

os.makedirs("./temp", exist_ok=True)


In [ ]:
# Release model and example data paths.
# Replace <pathto> with the location of the downloaded release bundle.
MODEL_ROOT = "<pathto>/release_model/SHREC"
DATA_ROOT = "<pathto>/release_model/data_example"

# Both detector and classifier use the native 512^3 volume.
TOMO_PATH = os.path.join(DATA_ROOT, "shrec.mrc")
CLASS_MASK_PATH = os.path.join(DATA_ROOT, "shrec_class_mask.mrc")
LABELS_PATH = os.path.join(DATA_ROOT, "shrec_labels.txt")

# Class mapping (SHREC 2021 official encoding).
CLASS_TO_ID = {
    "4V94": 1, "4CR2": 2, "1QVR": 3, "1BXN": 4, "3CF3": 5,
    "1U6G": 6, "3D2F": 7, "2CG9": 8, "3H84": 9, "3GL1": 10,
    "3QM1": 11, "1S3X": 12, "5MRC": 13,
}
SMALL_CLASSES = ["1S3X", "3QM1", "3GL1", "3H84", "2CG9", "1U6G", "3D2F"]
LARGE_CLASSES = ["5MRC", "4V94", "3CF3", "1BXN", "4CR2", "1QVR"]

# Inference settings.
DETECTOR_THRESHOLD = 0.20
SWEEP_DISTANCE = 15.0
CROP_SIZE = 41
UNIFORM_THRESHOLD = 0.40
Z_MARGIN = 10
Z_SIZE = 512
XY_SIZE = 512


## 2) Load Released Model

Load the paired detector (`last.ckpt`) and 3D CNN classifier (`3DCNN.ckpt`) for the small and large particle groups.

In [25]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# Load detector and classifier for each group.
models = {}
for group_name in ["small", "large"]:
    group_dir = os.path.join(MODEL_ROOT, group_name)
    detector = utils.loadModel(group_dir, "last.ckpt").to(device).eval()
    classifier = modules.Particle3DNet.load_from_checkpoint(
        os.path.join(group_dir, "3DCNN.ckpt"),
        map_location="cpu",
        strict=True,
    ).to(device).eval()
    models[group_name] = {"detector": detector, "classifier": classifier}
    print(f"{group_name}: detector and classifier loaded")


model at stage  stage 1
model with output classes 2
model receiving class weights tensor([1.0000, 0.2000])
using consistency regularization coef 0.5


/data/biosoftware/miniconda3/miniconda3/envs/tomognn/lib/python3.11/site-packages/torch/nn/modules/transformer.py:307: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(f"enable_nested_tensor is True, but self.use_nested_tensor is False because {why_not_sparsity_fast_path}")
/home/feity/cryoem/notebooks/../src/utils.py:1409: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will

incompatible parameters ['gnn.pe']
finish loading parameters
small: detector and classifier loaded
model at stage  stage 1
model with output classes 2
model receiving class weights tensor([1.0000, 0.2000])
using consistency regularization coef 0.5
incompatible parameters ['gnn.pe']
finish loading parameters
large: detector and classifier loaded


## 3) Load Example Data

Load the example tomogram and ground-truth labels. The tomogram is 512³; the class mask has the same shape with values 0–15 (0=background, 1–13=particles, 14=vesicle, 15=fiducial).

In [26]:
import mrcfile
from skimage.morphology import dilation

# Load tomogram for detector and classifier (both use the native 512^3 volume).
tomo_dataset = data.TestDatasetMrc(
    TOMO_PATH,
    norm="hist",
    reshape=800,
    length_for_average=3,
    gap=1,
)

# Load class mask and dilate it (official SHREC rule: 3x3x3 greyscale dilation).
with mrcfile.open(CLASS_MASK_PATH, permissive=True) as mrc:
    class_mask = mrc.data.copy()
dilated_class_mask = dilation(class_mask)

# Load ground-truth labels.
labels_df = pd.read_csv(LABELS_PATH, sep=r"\s+", header=None)
labels_df.columns = ["particle_name", "x", "y", "z", "_1", "_2", "_3"]
labels_df = labels_df[labels_df["particle_name"].isin(CLASS_TO_ID)].copy()
labels_df["class_id"] = labels_df["particle_name"].map(CLASS_TO_ID).astype(int)

print(f"Tomogram shape: {tomo_dataset.ori_mrc.shape}")
print(f"Class mask shape: {class_mask.shape}")
print(f"Ground-truth particles: {len(labels_df)}")


loading test dataset


/data/biosoftware/miniconda3/miniconda3/envs/tomognn/lib/python3.11/site-packages/mrcfile/mrcinterpreter.py:206: RuntimeWarning: Map ID string not found - not an MRC file, or file is corrupt
  warnings.warn(msg, RuntimeWarning)
/data/biosoftware/miniconda3/miniconda3/envs/tomognn/lib/python3.11/site-packages/mrcfile/mrcinterpreter.py:216: RuntimeWarning: Unrecognised machine stamp: 0x00 0x00 0x00 0x00
  warnings.warn(str(err), RuntimeWarning)


Tomogram shape: (512, 512, 512)
Class mask shape: (512, 512, 512)
Ground-truth particles: 1684


## 4) Run Prediction

Run the two-stage pipeline: detector proposes candidates, then the 3D CNN classifies each 41³ crop and refines its center via weighted-average decoding of the center-head logits.

In [28]:
def center_logits_to_crop_coords(center_logits, crop_size, method="weighted_average"):
    b, _, d, h, w = center_logits.shape
    flat_logits = center_logits.flatten(1)
    if method == "argmax":
        indices = flat_logits.argmax(dim=1)
        coords = torch.stack(
            [indices // (h * w), (indices % (h * w)) // w, indices % w], dim=1
        ).to(dtype=torch.float32)
    elif method == "weighted_average":
        probs = torch.softmax(flat_logits, dim=1)
        zz, yy, xx = torch.meshgrid(
            torch.arange(d, device=center_logits.device, dtype=probs.dtype),
            torch.arange(h, device=center_logits.device, dtype=probs.dtype),
            torch.arange(w, device=center_logits.device, dtype=probs.dtype),
            indexing="ij",
        )
        grid = torch.stack([zz, yy, xx], dim=-1).reshape(1, d * h * w, 3)
        coords = (probs.unsqueeze(-1) * grid).sum(dim=1)
    else:
        raise ValueError("method must be 'argmax' or 'weighted_average'")
    scale = torch.tensor(
        [(crop_size - 1) / max(d - 1, 1), (crop_size - 1) / max(h - 1, 1), (crop_size - 1) / max(w - 1, 1)],
        device=center_logits.device,
        dtype=coords.dtype,
    )
    return coords * scale


def run_group(group_name, particle_classes, detector_column):
    """Run detector + classifier for one particle group."""
    detector = models[group_name]["detector"]
    classifier = models[group_name]["classifier"]

    # Stage 1: detector proposals.
    raw = postprocess.generatedfBySlice(
        detector, tomo_dataset, gap=1, columns=[detector_column, "None"],
        num_classes=2, z_multiply=Z_SIZE,
    )
    raw = raw[raw[detector_column] >= DETECTOR_THRESHOLD].copy()
    raw["x"] = raw["x"] * XY_SIZE
    raw["y"] = raw["y"] * XY_SIZE
    points = raw[["z", "y", "x"]].to_numpy(dtype=float)
    scores = raw[detector_column].to_numpy(dtype=float)
    points, scores = utils.sweep_filter_points_by_distance(points, scores, threshold=SWEEP_DISTANCE)
    if len(points) == 0:
        return pd.DataFrame()
    keep = (
        (points[:, 0] >= Z_MARGIN) & (points[:, 0] < Z_SIZE - Z_MARGIN)
        & (points[:, 1] >= 0) & (points[:, 1] < XY_SIZE)
        & (points[:, 2] >= 0) & (points[:, 2] < XY_SIZE)
    )
    candidates = pd.DataFrame(points[keep], columns=["z", "y", "x"])
    candidates["stage1_score"] = scores[keep]

    # Stage 2: classifier on 41^3 crops.
    infer_df = candidates[["z", "y", "x"]].copy()
    infer_df["label"] = 0
    infer_df["tomogram"] = "shrec"
    ds = data.Particle3DDataset(
        infer_df, {"shrec": TOMO_PATH}, crop_size=CROP_SIZE,
        norm="hist", r=0, if_augmentation=False, output_center=True,
    )
    loader = torch.utils.data.DataLoader(ds, batch_size=32, shuffle=False, num_workers=0)

    probs_all = []
    centers_all = []
    with torch.inference_mode():
        for crops, _, _ in loader:
            crops = crops.to(device)
            logits, center_ret = classifier(crops, return_center=True)
            probs_all.append(torch.softmax(logits, dim=1).cpu().numpy())
            centers_all.append(
                center_logits_to_crop_coords(
                    center_ret["center_logits"], CROP_SIZE, method="weighted_average"
                ).cpu().numpy()
            )
    probs = np.concatenate(probs_all, axis=0)
    centers_crop = np.concatenate(centers_all, axis=0)

    # Map crop centers back to full-volume coordinates.
    centers_int = candidates[["z", "y", "x"]].astype(int).to_numpy()
    half = CROP_SIZE // 2
    origin = np.maximum(centers_int - half, 0)
    upper = np.array(ds.loaded_volumes["shrec"].shape) - 1
    revised = np.clip(origin + centers_crop, 0, upper)

    # Assign predicted class and score.
    class_ids = [CLASS_TO_ID[c] for c in particle_classes]
    group_probs = probs[:, [0] + class_ids]
    pred_idx = group_probs.argmax(axis=1)
    pred_class = ["background"] + particle_classes
    predicted_class = [pred_class[i] for i in pred_idx]
    prediction_score = group_probs.max(axis=1)

    out = candidates.copy()
    out["group"] = group_name
    out[["revised_z", "revised_y", "revised_x"]] = revised
    out["predicted_class"] = predicted_class
    out["prediction_score"] = prediction_score
    for name in particle_classes:
        out[f"score_{name}"] = probs[:, CLASS_TO_ID[name]]
    return out


predictions_small = run_group("small", SMALL_CLASSES, "small_particles")
predictions_large = run_group("large", LARGE_CLASSES, "large_particles")
predictions_df = pd.concat([predictions_small, predictions_large], ignore_index=True)

print(f"Small: {len(predictions_small)} candidates, Large: {len(predictions_large)} candidates")
print(f"Total: {len(predictions_df)} classified centers")
display(predictions_df.head())


100%|██████████| 512/512 [00:48<00:00, 10.50it/s]


finish stage1


Stage 2 slices: 100%|██████████| 512/512 [01:17<00:00,  6.64it/s]
/data/biosoftware/miniconda3/miniconda3/envs/tomognn/lib/python3.11/site-packages/mrcfile/mrcinterpreter.py:206: RuntimeWarning: Map ID string not found - not an MRC file, or file is corrupt
  warnings.warn(msg, RuntimeWarning)
/data/biosoftware/miniconda3/miniconda3/envs/tomognn/lib/python3.11/site-packages/mrcfile/mrcinterpreter.py:216: RuntimeWarning: Unrecognised machine stamp: 0x00 0x00 0x00 0x00
  warnings.warn(str(err), RuntimeWarning)
100%|██████████| 512/512 [00:46<00:00, 10.92it/s]


finish stage1


Stage 2 slices: 100%|██████████| 512/512 [00:56<00:00,  9.06it/s]
/data/biosoftware/miniconda3/miniconda3/envs/tomognn/lib/python3.11/site-packages/mrcfile/mrcinterpreter.py:206: RuntimeWarning: Map ID string not found - not an MRC file, or file is corrupt
  warnings.warn(msg, RuntimeWarning)
/data/biosoftware/miniconda3/miniconda3/envs/tomognn/lib/python3.11/site-packages/mrcfile/mrcinterpreter.py:216: RuntimeWarning: Unrecognised machine stamp: 0x00 0x00 0x00 0x00
  warnings.warn(str(err), RuntimeWarning)


Small: 2015 candidates, Large: 1602 candidates
Total: 3617 classified centers


,z,y,x,stage1_score,group,revised_z,revised_y,revised_x,predicted_class,prediction_score,...,score_3H84,score_2CG9,score_1U6G,score_3D2F,score_5MRC,score_4V94,score_3CF3,score_1BXN,score_4CR2,score_1QVR
0,10.0,81.200729,231.838287,0.214812,small,20.000092,80.999918,230.999893,background,0.999417,...,1.613219e-07,3.458535e-08,1.132109e-09,7.246653e-08,NaN,NaN,NaN,NaN,NaN,NaN
1,14.0,487.423401,182.553665,0.218924,small,20.000000,487.000000,182.000000,background,0.999979,...,1.797790e-06,1.039544e-07,5.771428e-09,1.476591e-08,NaN,NaN,NaN,NaN,NaN,NaN
2,15.0,204.821091,235.554626,0.202799,small,20.000000,204.000000,235.000000,background,0.999956,...,2.790339e-06,1.860519e-07,4.470860e-09,2.397201e-08,NaN,NaN,NaN,NaN,NaN,NaN
3,16.0,91.319740,139.677124,0.215944,small,20.000000,91.000000,139.000000,background,0.999842,...,9.225877e-06,1.493365e-06,1.417028e-08,7.678503e-07,NaN,NaN,NaN,NaN,NaN,NaN
4,24.0,442.523865,235.654160,0.210999,small,24.000000,442.000000,235.000000,background,0.978974,...,6.320659e-03,1.072921e-03,2.242521e-04,1.515536e-03,NaN,NaN,NaN,NaN,NaN,NaN


## 5) Compute SHREC Official F1 Score

Apply the uniform threshold and evaluate using the official-style mask hit rule: a prediction is a hit when its rounded coordinate lands on a dilated class-mask voxel for that particle. The mean per-class F1 is reported with and without 4V94 (whose labels are known to be buggy).

In [29]:
def evaluate_mask(predictions_frame, mask, particle_names, threshold):
    """Official-style SHREC mask evaluation.

    A prediction hits GT particle p when its rounded (x,y,z) lies on p's class
    region in the dilated mask and is closer to p's annotated center than to any
    other GT center of the same class. Localization is class-agnostic;
    classification assigns each found GT particle the label of its highest-
    scoring matched prediction.
    """
    particle_values = {CLASS_TO_ID[n] for n in particle_names}
    sub = predictions_frame[predictions_frame["prediction_score"] >= threshold].copy()
    sub = sub[sub["predicted_class"].isin(particle_names)].reset_index(drop=True)
    if len(sub) == 0:
        return pd.DataFrame(columns=["particle", "gt_count", "predicted", "found_gt", "precision", "recall", "f1"])

    coords = np.clip(np.round(sub[["revised_x", "revised_y", "revised_z"]].to_numpy()).astype(int), 0, 511)
    sub["hit_mask_value"] = mask[coords[:, 2], coords[:, 1], coords[:, 0]].astype(int)
    sub["is_hit"] = sub["hit_mask_value"].isin(particle_values)

    rows = []
    for name in particle_names:
        value = CLASS_TO_ID[name]
        gt_cls = labels_df[labels_df["particle_name"] == name].reset_index(drop=True)
        n_gt = int(len(gt_cls))
        eligible = sub[sub["is_hit"] & (sub["hit_mask_value"] == value)].copy()
        found_gt = 0
        if len(eligible) and n_gt:
            pred_pts = eligible[["revised_z", "revised_y", "revised_x"]].to_numpy(dtype=float)
            gt_pts = gt_cls[["z", "y", "x"]].to_numpy(dtype=float)
            d = np.linalg.norm(pred_pts[:, None, :] - gt_pts[None, :, :], axis=2)
            found_gt = int(len(np.unique(d.argmin(axis=1))))
        n_pred_cls = int((sub["predicted_class"] == name).sum())
        precision = found_gt / n_pred_cls if n_pred_cls else 0.0
        recall = found_gt / n_gt if n_gt else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
        rows.append({
            "particle": name, "gt_count": n_gt, "predicted": n_pred_cls,
            "found_gt": found_gt, "precision": precision, "recall": recall, "f1": f1,
        })
    return pd.DataFrame(rows)


# Evaluate with and without 4V94.
for label, classes in [("with 4V94", list(CLASS_TO_ID)), ("without 4V94", [c for c in CLASS_TO_ID if c != "4V94"])]:
    df = evaluate_mask(predictions_df, dilated_class_mask, classes, UNIFORM_THRESHOLD)
    mean_f1 = df["f1"].mean()
    print(f"Mean per-class F1 ({label}): {mean_f1:.4f}")
    display(df)


Mean per-class F1 (with 4V94): 0.7804


,particle,gt_count,predicted,found_gt,precision,recall,f1
0,4V94,130,181,114,0.629834,0.876923,0.733119
1,4CR2,115,116,113,0.974138,0.982609,0.978355
2,1QVR,127,117,109,0.931624,0.858268,0.893443
3,1BXN,135,119,114,0.957983,0.844444,0.897638
4,3CF3,139,120,121,1.008333,0.870504,0.934363
5,1U6G,143,105,101,0.961905,0.706294,0.814516
6,3D2F,140,112,105,0.937500,0.750000,0.833333
7,2CG9,125,110,98,0.890909,0.784000,0.834043
8,3H84,144,112,102,0.910714,0.708333,0.796875
9,3GL1,123,127,90,0.708661,0.731707,0.720000


Mean per-class F1 (without 4V94): 0.7843


,particle,gt_count,predicted,found_gt,precision,recall,f1
0,4CR2,115,116,113,0.974138,0.982609,0.978355
1,1QVR,127,117,109,0.931624,0.858268,0.893443
2,1BXN,135,119,114,0.957983,0.844444,0.897638
3,3CF3,139,120,121,1.008333,0.870504,0.934363
4,1U6G,143,105,101,0.961905,0.706294,0.814516
5,3D2F,140,112,105,0.937500,0.750000,0.833333
6,2CG9,125,110,98,0.890909,0.784000,0.834043
7,3H84,144,112,102,0.910714,0.708333,0.796875
8,3GL1,123,127,90,0.708661,0.731707,0.720000
9,3QM1,120,108,67,0.620370,0.558333,0.587719
